<a href="https://colab.research.google.com/github/carabasroxana/psy_detectives/blob/main/Another_copy_of_TinyLlama_1_1B_Chat_v1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import pandas as pd
import json

DATA_PATH = "/content/train_rehydrated.jsonl"

df = pd.read_json(DATA_PATH, lines=True)
df[["conspiracy"]].value_counts()
df["conspiracy"].unique()

array(['Yes', "Can't tell", 'No'], dtype=object)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
import re

GOLD_MAP = {
    "Yes": "Yes",
    "No": "No",
    "Can't tell": "Cant",
    "Cant": "Cant",
    "Can\'t tell": "Cant"
}

VALID_LABELS = {"Yes", "No", "Cant"}

def build_prompt(text: str) -> str:
    """
    Build a short instruction-style prompt for TinyLlama.
    """
    return f"""### Instruction:
You are an expert annotator of online conspiracy content.
Your task is to decide if the following Reddit post expresses a conspiracy belief.

Label it with exactly one of:
- Yes  (the post clearly endorses or presents a conspiracy explanation)
- No   (no conspiracy reasoning is present)
- Cant (unclear or not enough information)

Post:
{text}

### Response:
Answer with only one word: Yes, No, or Cant.
"""

def parse_label(generated_text: str) -> str:
    """
    Extract Yes/No/Cant from the LLM output.
    """
    text = generated_text.strip().lower()
    if "yes" in text:
        return "Yes"
    if "cant" in text or "can't" in text:
        return "Cant"
    if "no" in text:
        return "No"
    last = text.split()[-1].capitalize() if text.split() else ""
    return last if last in VALID_LABELS else "Cant"

In [ ]:
@torch.no_grad()
def llm_predict_label(post_text: str) -> str:
    prompt = build_prompt(post_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False
    )
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
    return parse_label(generated)

In [ ]:
example_text = df["text"].iloc[0]
print(example_text)
print("LLM prediction:", llm_predict_label(example_text))

A great article on what's taking place in Bolivia, referencing some similar US backed coups in the region as well as recounting some of Bolivia's history and western policy towards the country.
LLM prediction: Yes


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

df_eval = df[df["conspiracy"].isin(GOLD_MAP.keys())].copy()

N = 500
df_sample = df_eval.sample(min(N, len(df_eval)), random_state=42).reset_index(drop=True)

gold = []
pred = []

for _, row in df_sample.iterrows():
    text = row["text"]
    true_label_raw = row["conspiracy"]
    true_label = GOLD_MAP.get(true_label_raw, "Cant")

    predicted_label = llm_predict_label(text)

    gold.append(true_label)
    pred.append(predicted_label)

print("Number of examples:", len(gold))
print("Accuracy:", accuracy_score(gold, pred))

print("\nGold label distribution:")
print(pd.Series(gold).value_counts())

print("\nPredicted label distribution:")
print(pd.Series(pred).value_counts())

print("\nClassification report (zero_division=0):\n")
print(classification_report(gold, pred, labels=["Yes","No","Cant"], zero_division=0))

print("\nConfusion matrix (rows=gold, cols=pred):")
print(confusion_matrix(gold, pred, labels=["Yes","No","Cant"]))